In [9]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print("\nFolder:", root)
    for file in files[:10]:
        print("File:", file)


Folder: /kaggle/input

Folder: /kaggle/input/competitions

Folder: /kaggle/input/competitions/shl-hiring-assessment-2026

Folder: /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final
File: sample_submission.csv
File: train.csv
File: test.csv

Folder: /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test
File: audio_49.wav
File: audio_90.wav
File: audio_77.wav
File: audio_66.wav
File: audio_54.wav
File: audio_42.wav
File: audio_81.wav
File: audio_72.wav
File: audio_107.wav
File: audio_79.wav

Folder: /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/train
File: audio_49.wav
File: audio_663.wav
File: audio_5059.wav
File: audio_90.wav
File: audio_581.wav
File: audio_698.wav
File: audio_77.wav
File: audio_66.wav
File: audio_555.wav
File: audio_54.wav


In [10]:
import os
import pandas as pd

# Dataset location
DATA_DIR = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

# Load CSV files
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_submission = pd.read_csv(
    os.path.join(DATA_DIR, "sample_submission.csv")
)

# Display dataset information
print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)

print("\nTraining data:")
display(train_df.head())

print("\nTesting data:")
display(test_df.head())

print("\nSample submission format:")
display(sample_submission.head())

print("\nTraining columns:", train_df.columns.tolist())
print("Testing columns:", test_df.columns.tolist())

Training data shape: (769, 2)
Testing data shape: (216, 2)

Training data:


,filename,label
0,audio_192.wav,3.0
1,audio_436.wav,3.0
2,audio_447.wav,3.5
3,audio_126.wav,2.0
4,audio_61.wav,2.0



Testing data:


,filename,label
0,audio_128.wav,-1
1,audio_156.wav,-1
2,audio_19.wav,-1
3,audio_108.wav,-1
4,audio_206.wav,-1



Sample submission format:


,filename,label
0,audio_804.wav,-1
1,audio_1028.wav,-1
2,audio_865.wav,-1
3,audio_774.wav,-1
4,audio_1138.wav,-1



Training columns: ['filename', 'label']
Testing columns: ['filename', 'label']


In [11]:
import os
import numpy as np
import pandas as pd
import scipy
from scipy.io import wavfile
from scipy.signal import resample_poly
from tqdm.auto import tqdm

DATA_DIR = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

TRAIN_AUDIO_DIR = os.path.join(DATA_DIR, "train")
TEST_AUDIO_DIR = os.path.join(DATA_DIR, "test")

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

def extract_features(file_path):
    try:
        sr, audio = wavfile.read(file_path)

        # Convert stereo audio to mono
        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        audio = audio.astype(np.float32)

        # Normalize audio
        peak = np.max(np.abs(audio))
        if peak > 0:
            audio = audio / peak

        # Reduce the sample rate to speed up processing
        if sr > 8000:
            audio = resample_poly(audio, 8000, sr)
            sr = 8000

        # Split audio into short frames
        frame_size = 4000
        usable = (len(audio) // frame_size) * frame_size

        if usable == 0:
            return np.zeros(12)

        frames = audio[:usable].reshape(-1, frame_size)

        # Calculate simple audio statistics
        rms = np.sqrt(np.mean(frames ** 2, axis=1))
        zcr = np.mean(
            np.signbit(frames[:, 1:]) != np.signbit(frames[:, :-1]),
            axis=1
        )

        spectrum = np.abs(np.fft.rfft(frames, axis=1))
        freqs = np.fft.rfftfreq(frame_size, 1 / sr)

        spectral_centroid = (
            (spectrum * freqs).sum(axis=1)
            / (spectrum.sum(axis=1) + 1e-8)
        )

        features = [
            len(audio) / sr,
            np.mean(audio),
            np.std(audio),
            np.mean(rms),
            np.std(rms),
            np.mean(zcr),
            np.std(zcr),
            np.mean(spectral_centroid),
            np.std(spectral_centroid),
            np.percentile(rms, 25),
            np.percentile(rms, 75),
            np.max(rms)
        ]

        return np.nan_to_num(np.array(features))

    except Exception as e:
        print("Error:", file_path, e)
        return np.zeros(12)


# Extract features from training recordings
X_train = np.array([
    extract_features(os.path.join(TRAIN_AUDIO_DIR, name))
    for name in tqdm(train_df["filename"], desc="Training audio")
])

# Extract features from testing recordings
X_test = np.array([
    extract_features(os.path.join(TEST_AUDIO_DIR, name))
    for name in tqdm(test_df["filename"], desc="Testing audio")
])

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)
print("Feature extraction completed!")

Training audio:   0%|          | 0/769 [00:00<?, ?it/s]

Testing audio:   0%|          | 0/216 [00:00<?, ?it/s]

Training features shape: (769, 12)
Testing features shape: (216, 12)
Feature extraction completed!


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

# Target grammar scores
y = train_df["label"].astype(float)

# Split training data into training and validation sets
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y,
    test_size=0.20,
    random_state=42
)

# Build a machine learning model
model = make_pipeline(
    SimpleImputer(strategy="median"),
    ExtraTreesRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
)

# Train the model
model.fit(X_tr, y_tr)

# Predict validation scores
val_predictions = model.predict(X_val)

# Calculate evaluation metrics
rmse = np.sqrt(mean_squared_error(y_val, val_predictions))
mae = mean_absolute_error(y_val, val_predictions)
r2 = r2_score(y_val, val_predictions)

print("Model training completed!")
print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation MAE: {mae:.4f}")
print(f"Validation R2 Score: {r2:.4f}")

In [ ]:
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# Calculate training RMSE for the existing model
train_predictions = model.predict(X_tr)

train_rmse = np.sqrt(
    mean_squared_error(y_tr, train_predictions)
)

print(f"Training RMSE: {train_rmse:.4f}")
print(f"Validation RMSE: {rmse:.4f}")

# Train final model using all 769 training samples
final_model = clone(model)
final_model.fit(X_train, y)

# Predict grammar scores for 216 test recordings
test_predictions = final_model.predict(X_test)

# Keep scores within the allowed range
test_predictions = np.clip(test_predictions, 0, 5)

# Prepare submission using the official template
submission = sample_submission.copy()

prediction_map = dict(
    zip(test_df["filename"], test_predictions)
)

submission["label"] = submission["filename"].map(prediction_map)

# Verify submission
assert submission["label"].notna().all()
assert len(submission) == len(test_df)

# Save submission file
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("Submission file created successfully!")
print("Total predictions:", len(submission))

display(submission.head())

In [12]:
import os
import pandas as pd

test_path = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test.csv"
submission_path = "/kaggle/working/submission.csv"

test_df = pd.read_csv(test_path)

if os.path.exists(submission_path):
    submission = pd.read_csv(submission_path)

    print("Test rows:", len(test_df))
    print("Submission rows:", len(submission))
    print("Missing labels:", submission["label"].isna().sum())
    print(submission.head())
else:
    print("submission.csv not found.")

Test rows: 216
Submission rows: 216
Missing labels: 0
        filename     label
0  audio_128.wav  3.066667
1  audio_156.wav  3.405972
2   audio_19.wav  3.828056
3  audio_108.wav  3.046222
4  audio_206.wav  2.666389


In [13]:
import pandas as pd
import numpy as np

submission = pd.read_csv("/kaggle/working/submission.csv")
test_df = pd.read_csv(
    "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test.csv"
)

print("Submission columns:", submission.columns.tolist())
print("Test columns:", test_df.columns.tolist())
print("Rows match:", len(submission) == len(test_df))
print("Missing labels:", submission["label"].isna().sum())
print("All labels are numeric:", 
      pd.to_numeric(submission["label"], errors="coerce").notna().all())
print("Minimum predicted score:", submission["label"].min())
print("Maximum predicted score:", submission["label"].max())
print("Duplicate filenames:", submission["filename"].duplicated().sum())

Submission columns: ['filename', 'label']
Test columns: ['filename', 'label']
Rows match: True
Missing labels: 0
All labels are numeric: True
Minimum predicted score: 2.255416666666668
Maximum predicted score: 4.580277777777778
Duplicate filenames: 0


In [14]:
# Verify filename matching

test_files = set(test_df["filename"])
submission_files = set(submission["filename"])

print("Test filenames:", len(test_files))
print("Submission filenames:", len(submission_files))

print("Missing filenames:", len(test_files - submission_files))
print("Extra filenames:", len(submission_files - test_files))

print("All filenames match:", test_files == submission_files)

Test filenames: 216
Submission filenames: 216
Missing filenames: 0
Extra filenames: 0
All filenames match: True


In [ ]:
# Check the filenames in the test and submission files

print("Test rows:", len(test_df))
print("Sample submission rows:", len(sample_submission))

print("\nTest CSV columns:", test_df.columns.tolist())
print("Submission columns:", sample_submission.columns.tolist())

print("\nTest filenames:")
print(test_df["filename"].head())

print("\nSubmission filenames:")
print(sample_submission["filename"].head())

print("\nMatching filenames:",
      sample_submission["filename"].isin(test_df["filename"]).sum())

In [ ]:
import os
import numpy as np
import pandas as pd

# Train the final model using all training data
final_model.fit(X_train, train_df["label"].astype(float))

# Predict scores for all 216 test audio files
predictions = final_model.predict(X_test)

# Keep scores within the allowed range
predictions = np.clip(predictions, 0, 5)

# Build submission directly from test.csv
submission = pd.DataFrame({
    "filename": test_df["filename"].values,
    "label": predictions
})

# Verify submission
print("Test rows:", len(test_df))
print("Submission rows:", len(submission))
print("Missing predictions:", submission["label"].isna().sum())
print("\nFirst five predictions:")
display(submission.head())

assert len(submission) == 216
assert submission["label"].notna().all()

# Save the file
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("\nSUCCESS! submission.csv has been created.")